# Research, read the answer, execute the saved study

Run the cells from top to bottom using **Epoch SDK (local prototype)**.

We supply two synthetic observations: **2024 = 10 USD**, **2025 = 15 USD**.
The expected answer has a two-row table, JSON with `row_count=2` and `total_sales=25`,
and a short text explanation. These are software demonstration values.

The research cell starts real EpochAI/model and Backend work. A saved-study execution
is a separate action. Re-running the same operation ID and inputs attaches to the
same work. Choose a new ID when you want a new task or different parameters.

## 1. Connect to the running services

The prepared notebook server supplies your Epoch identity and both service URLs. Credentials are not stored in this notebook.

In [ ]:
from IPython.display import JSON, display
from stratifyx import DatePeriodConfig
from epoch_sdk.research import ResearchCompleted
from epoch_sdk.supervisor import NeedsInput
from notebook_support import (
    connect, check_services, demo_request, operation_name,
    wait_research, wait_execution, show_answer, check_demo_values,
)

epoch = connect()
await check_services()

## 2. Describe the task and requested answer

Open the JSON below to see the natural-language task, supplied context and annotated table/JSON/text declarations. Change the operation ID as well when changing accepted inputs.

In [ ]:
request = demo_request()
RESEARCH_ID = operation_name("research")
EXECUTION_ID = operation_name("execute")
print("Research ID:", RESEARCH_ID)
print("Execution ID:", EXECUTION_ID)
display(JSON(request.to_wire(), expanded=False))

## 3. Research: build, execute, inspect and publish

This may take a minute. The returned answer is saved by Backend. If the model asks a question, inspect it before continuing; notebook 02 demonstrates answering and resuming.

In [ ]:
result = await epoch.research(request, operation_id=RESEARCH_ID)
if not isinstance(result, (ResearchCompleted, NeedsInput)):
    result = await wait_research(epoch, RESEARCH_ID)
if isinstance(result, NeedsInput):
    print(result.feedback())
else:
    print("Research complete:", result.response.response_id)
    print("Reusable study:", result.reusable_study.study_id if result.reusable_study else None)

## 4. Read the published table, JSON and text

The table comes from saved computed data. Displaying it does not request another dashboard or start another study.

In [ ]:
assert isinstance(result, ResearchCompleted), "Answer the pending question before reading a completed answer."
await show_answer(epoch, result)

## 5. Verify all rows, including pagination

A page size of one deliberately exercises both pages. The total table is not limited to a single transfer.

In [ ]:
rows = []
async for page in epoch.iter_response_table(
    result.response.response_id, "annual_sales",
    version=result.response.version, page_size=1,
):
    rows.extend(page.rows)
    print("Page offset:", page.offset, "next:", page.next_offset)
check_demo_values(result, rows)

## 6. Execute the saved study directly through Backend

This reuses the prepared study and creates one new execution for this execution operation ID. It does not repeat discovery or ask the supervisor to rebuild the study.

In [ ]:
assert result.reusable_study is not None
period = DatePeriodConfig(start="2024-01-01", end="2025-12-31")
execution = await epoch.execute_saved(
    result.reusable_study,
    period=period,
    operation_id=EXECUTION_ID,
)
execution = await wait_execution(epoch, EXECUTION_ID)
assert execution.execution.definition_id == result.reusable_study.study_id
print("Study:", execution.execution.definition_id)
print("New execution:", execution.execution.id)
print("Status:", execution.execution.status.value)

## 7. Repeat the call: attach to the same execution

Run this cell repeatedly. A new intended execution needs a new operation ID.

In [ ]:
same_execution = await epoch.execute_saved(
    result.reusable_study,
    period=period,
    operation_id=EXECUTION_ID,
)
assert same_execution.execution.id == execution.execution.id
print("Same execution:", same_execution.execution.id)

## 8. The research answer keeps its original source

Prepared execution returns an execution. The research answer remains pinned to its original run and revision. A new interpretation of new results is a separate answer-producing step.

In [ ]:
original = await epoch.research_status(operation_id=RESEARCH_ID)
assert isinstance(original, ResearchCompleted)
assert original.response.response_id == result.response.response_id
assert original.response.version == result.response.version
assert original.response.primary_execution_id != execution.execution.id
print("Original answer execution:", original.response.primary_execution_id)
print("Prepared execution:", execution.execution.id)
print("Pinned answer revision:", original.response.version)

## 9. Close this client

The services keep running. Notebook 02 exercises reconnecting from a fresh client and answering clarification.

In [ ]:
await epoch.aclose()